# SimpleQA Dataset Evaluation

In [1]:
%pip install -U transformers datasets peft accelerate bitsandbytes

In [2]:
import torch
import json
import time
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple
import ast
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from difflib import SequenceMatcher
import re
import pandas as pd
import numpy as np
from tqdm import tqdm

In [3]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

# %cd /kaggle/working/Active-Reading--Pattern-Recognition

%cd /content/Active-Reading--Pattern-Recognition # if in Colab

os.getcwd()

fatal: destination path 'Active-Reading--Pattern-Recognition' already exists and is not an empty directory.
[Errno 2] No such file or directory: '/content/Active-Reading--Pattern-Recognition # if in Colab'
/content


'/content'

## Load the SimpleQA Dataset

In [4]:
import json

def alnum_key(s: str) -> str:
    return "".join(ch.lower() for ch in (s or "") if ch.isalnum())

names_path = "/content/Active-Reading--Pattern-Recognition/filtered_doc_names_sw.json"

with open(names_path, "r", encoding="utf-8") as f:
    target_names = json.load(f)

target_keys = {alnum_key(n) for n in target_names}
print("target_keys:", len(target_keys))  # should be 856

target_keys: 856


In [5]:
import ast
from urllib.parse import urlparse, unquote

WIKI_HOSTS = {"en.wikipedia.org"}
WIKI_PREFIX = "/wiki/"

def parse_metadata(meta_str: str) -> dict:
    """metadata is stored as a string; parse into a dict safely."""
    if not meta_str:
        return {}
    try:
        # Many rows look like "{'topic':..., 'urls':[...]}": python-literal style
        return ast.literal_eval(meta_str)
    except Exception:
        # fallback: return empty
        return {}

def extract_wikipedia_title(url: str) -> str | None:
    """Return decoded Wikipedia page title (string after /wiki/), or None."""
    if not url:
        return None
    try:
        u = urlparse(url)
    except Exception:
        return None

    if u.netloc not in WIKI_HOSTS:
        return None
    if not u.path.startswith(WIKI_PREFIX):
        return None

    title = u.path[len(WIKI_PREFIX):]  # the part after /wiki/
    title = unquote(title)
    title = title.split("#", 1)[0].split("?", 1)[0]
    title = title.replace("_", " ").strip()
    return title if title else None

def filter_urls_to_target_wikipedia(urls: list[str], target_keys: set[str]) -> list[str]:
    """Keep only Wikipedia URLs whose titles match target_keys alphanumerically."""
    kept = []
    for url in urls or []:
        title = extract_wikipedia_title(url)
        if title and alnum_key(title) in target_keys:
            kept.append(url)
    return kept


In [6]:
from datasets import load_dataset

print("Loading SimpleQA...")
ds = load_dataset("basicv8vc/SimpleQA")
test = ds["test"]

def row_matches(example):
    meta = parse_metadata(example.get("metadata", ""))
    urls = meta.get("urls", [])
    kept_wiki = filter_urls_to_target_wikipedia(urls, target_keys)
    return len(kept_wiki) > 0

filtered = test.filter(row_matches)

print("Original test rows:", len(test))
print("Filtered rows:", len(filtered))

Loading SimpleQA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Original test rows: 4326
Filtered rows: 1252


In [7]:
SEED = 42
N_SAMPLES = 100

filtered_100 = filtered.shuffle(seed=SEED).select(range(N_SAMPLES))

print("Final evaluation set size:", len(filtered_100))

Final evaluation set size: 100


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Model Loading

In [9]:
# Model configuration
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
ADAPTER_PATH = "/content/drive/MyDrive/final_qlora_adapter_repetition"

print(f"Base model: {BASE_MODEL}")
print(f"Adapter path: {ADAPTER_PATH}")

Base model: Qwen/Qwen3-4B-Instruct-2507
Adapter path: /content/drive/MyDrive/final_qlora_adapter_repetition


In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

base_only_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
base_only_model.eval()

print("Model loaded successfully with adapter!")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded successfully with adapter!


## Evaluation Metrics

In [11]:
def validate_answer(model, tokenizer, question: str, expected_answer: str, predicted_answer: str) -> bool:
    """Use the model to validate if the predicted answer is correct."""
    validation_prompt = f"""You are an expert answer validator. Given a question, an expected answer, and a predicted answer, determine if the predicted answer is correct.

Question: {question}
Expected Answer: {expected_answer}
Predicted Answer: {predicted_answer}

Is the predicted answer correct? Answer with only YES or NO."""

    if tokenizer.chat_template is not None:
        messages = [{"role": "user", "content": validation_prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        )
    else:
        formatted_prompt = validation_prompt

    response = generate_text(
        model,
        tokenizer,
        prompt=formatted_prompt,
        max_tokens=16,
    )

    response = response.strip().upper()
    return "YES" in response

print("Model-based validator defined")

Model-based validator defined


## Generate Model Predictions

In [12]:
def generate_text(model, tokenizer, prompt: str, max_tokens: int = 256) -> str:
    """
    Generate text using transformers model.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

def transformers_generate_compat(model, tokenizer, prompt: str, max_tokens: int = 256) -> str:
    """
    Generate text using transformers model with chat template support.
    """
    if tokenizer.chat_template is not None:
        messages = [{"role": "user", "content": prompt}]
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        )

    return generate_text(model, tokenizer, prompt, max_tokens)

# Configuration for evaluation
NUM_SAMPLES = 100  # Evaluate on first 100 samples; set to -1 for all
MAX_TOKENS = 256
PROMPT_TEMPLATE = """Answer the following question with a concise answer.

Question: {question}

Answer:"""

print(f"Evaluation configuration:")
print(f"- Number of samples: {NUM_SAMPLES if NUM_SAMPLES > 0 else len(ds['test'])}")
print(f"- Max tokens per generation: {MAX_TOKENS}")

Evaluation configuration:
- Number of samples: 100
- Max tokens per generation: 256


In [14]:
# Run evaluation
results = []
num_eval_samples = NUM_SAMPLES if NUM_SAMPLES > 0 else len(filtered)

print(f"\nGenerating predictions on {num_eval_samples} samples...")
print("=" * 80)

for idx in tqdm(range(num_eval_samples), desc="Evaluating"):
    example = filtered[idx]
    question = example['problem']
    ground_truth = example['answer']
    metadata = ast.literal_eval(example['metadata'])

    # Generate prompt
    prompt = PROMPT_TEMPLATE.format(question=question)

    # Generate prediction
    start_time = time.time()
    prediction = transformers_generate_compat(model, tokenizer, prompt, MAX_TOKENS)
    generation_time = time.time() - start_time

    # Store results
    result = {
        'index': idx,
        'question': question,
        'ground_truth': ground_truth,
        'prediction': prediction,
        'topic': metadata.get('topic', 'Unknown'),
        'answer_type': metadata.get('answer_type', 'Unknown'),
        'generation_time': generation_time,
    }
    results.append(result)

print("=" * 80)
print(f"Predictions generated for {len(results)} samples")
print(f"Average generation time: {np.mean([r['generation_time'] for r in results]):.3f}s")


Generating predictions on 100 samples...


Evaluating:  16%|█▌        | 16/100 [01:18<06:51,  4.90s/it]


KeyboardInterrupt: 

## Results

In [14]:
# Compute evaluation metrics for each result
print("Validating predictions using model-based validator...")

for result in tqdm(results, desc="Validating"):
    question = result['question']
    ground_truth = result['ground_truth']
    prediction = result['prediction']

    is_correct = validate_answer(base_only_model, tokenizer, question, ground_truth, prediction)
    result['is_correct'] = int(is_correct)

# Convert to DataFrame for easier analysis
df_results = pd.DataFrame(results)

print("\n" + "=" * 80)
print("OVERALL EVALUATION RESULTS")
print("=" * 80)
print(f"\nTotal samples evaluated: {len(df_results)}")
print(f"\nMetrics Summary:")
print(f"  Accuracy (Model-Validated): {df_results['is_correct'].mean():.4f} ({df_results['is_correct'].sum()}/{len(df_results)})")
print(f"  Avg Generation Time: {df_results['generation_time'].mean():.3f}s")

Validating predictions using model-based validator...


Validating: 100%|██████████| 100/100 [00:24<00:00,  4.16it/s]


OVERALL EVALUATION RESULTS

Total samples evaluated: 100

Metrics Summary:
  Accuracy (Model-Validated): 0.1700 (17/100)
  Avg Generation Time: 4.189s


In [15]:
# Save results to file
output_file = Path("evaluation_results.jsonl")
print(f"\nSaving results to {output_file}...")

with open(output_file, 'w') as f:
    for result in results:
        f.write(json.dumps(result) + '\n')

print(f"Results saved to {output_file}")


Saving results to evaluation_results.jsonl...
Results saved to evaluation_results.jsonl


# Gold Context - Model with adapters

In [19]:
import json, time, ast
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

WIKI_CORPUS_JSON = "/content/Active-Reading--Pattern-Recognition/Datasets/simple_wiki_corpus.json"  # <-- CHANGE

with open(WIKI_CORPUS_JSON, "r", encoding="utf-8") as f:
    corpus_rows = json.load(f)

corpus_index = {}
for r in corpus_rows:
    dn = r.get("doc_name")
    tx = r.get("text")
    if isinstance(dn, str) and isinstance(tx, str) and tx.strip():
        corpus_index[alnum_key(dn)] = tx.strip()

print(f"Corpus loaded: {len(corpus_rows)} rows | indexed: {len(corpus_index)} docs")


def generate_text(model, tokenizer, prompt: str, max_tokens: int = 256) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

def transformers_generate_compat(model, tokenizer, prompt: str, max_tokens: int = 256) -> str:
    if tokenizer.chat_template is not None:
        messages = [{"role": "user", "content": prompt}]
        prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return generate_text(model, tokenizer, prompt, max_tokens)

def validate_answer(model, tokenizer, question: str, expected_answer: str, predicted_answer: str) -> bool:
    validation_prompt = f"""You are an expert answer validator. Given a question, an expected answer, and a predicted answer, determine if the predicted answer is correct.

Question: {question}
Expected Answer: {expected_answer}
Predicted Answer: {predicted_answer}

Is the predicted answer correct? Answer with only YES or NO."""
    if tokenizer.chat_template is not None:
        messages = [{"role": "user", "content": validation_prompt}]
        formatted_prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    else:
        formatted_prompt = validation_prompt

    response = generate_text(model, tokenizer, formatted_prompt, max_tokens=16).strip().upper()
    return "YES" in response

print("Model-based validator defined")

def build_gold_context(ex, max_items=3, include_titles=True):
    meta = parse_metadata(ex.get("metadata", ""))
    urls = meta.get("urls", []) or []

    kept_urls = filter_urls_to_target_wikipedia(urls, target_keys)

    parts = []
    for url in kept_urls:
        title = extract_wikipedia_title(url)
        if not title:
            continue

        tx = corpus_index.get(alnum_key(title))
        if not tx:
            continue

        parts.append(f"### {title}\n{tx}" if include_titles else tx)

        if len(parts) >= max_items:
            break

    return "\n\n---\n\n".join(parts)

GOLD_PROMPT = """Use ONLY the evidence below to answer the question.
Answer with the exact value/text from the evidence. Do not add extra explanation.
If the evidence does not contain the answer, output exactly: I don't know.

Evidence:
{context}

Question: {question}
Answer:"""


EVAL_SET = filtered_100 if "filtered_100" in globals() else filtered

NUM_SAMPLES = 100 if "NUM_SAMPLES" not in globals() else NUM_SAMPLES
MAX_TOKENS  = 256 if "MAX_TOKENS"  not in globals() else MAX_TOKENS

results = []
num_eval_samples = min(NUM_SAMPLES, len(EVAL_SET)) if NUM_SAMPLES > 0 else len(EVAL_SET)

print(f"\nGenerating GOLD-CONTEXT predictions on {num_eval_samples} samples...")
print("=" * 80)

for idx in tqdm(range(num_eval_samples), desc="Evaluating (Gold Context)"):
    example = EVAL_SET[idx]

    question = example["problem"]      # your notebook key
    ground_truth = example["answer"]
    meta = parse_metadata(example.get("metadata", ""))

    context = build_gold_context(example, max_items=3, include_titles=True)
    prompt = GOLD_PROMPT.format(context=context, question=question)

    start_time = time.time()
    prediction = transformers_generate_compat(model, tokenizer, prompt, MAX_TOKENS)
    generation_time = time.time() - start_time

    results.append({
        "index": idx,
        "question": question,
        "ground_truth": ground_truth,
        "prediction": prediction,
        "topic": meta.get("topic", "Unknown"),
        "answer_type": meta.get("answer_type", "Unknown"),
        "generation_time": generation_time,
        "context_len_chars": len(context),
    })

print("=" * 80)
print(f"Predictions generated for {len(results)} samples")
print(f"Average generation time: {np.mean([r['generation_time'] for r in results]):.3f}s")

print("Validating predictions using model-based validator...")

for r in tqdm(results, desc="Validating"):
    r["is_correct"] = int(validate_answer(base_only_model, tokenizer, r["question"], r["ground_truth"], r["prediction"]))

df_results = pd.DataFrame(results)

print("\n" + "=" * 80)
print("OVERALL GOLD-CONTEXT EVALUATION RESULTS")
print("=" * 80)
print(f"Total samples evaluated: {len(df_results)}")
print(f"Accuracy (Model-Validated): {df_results['is_correct'].mean():.4f} ({df_results['is_correct'].sum()}/{len(df_results)})")
print(f"Avg Generation Time: {df_results['generation_time'].mean():.3f}s")
print(f"Avg Context Length (chars): {df_results['context_len_chars'].mean():.1f}")

Corpus loaded: 4020 rows | indexed: 2810 docs
Model-based validator defined

Generating GOLD-CONTEXT predictions on 100 samples...


Evaluating (Gold Context): 100%|██████████| 100/100 [04:02<00:00,  2.43s/it]


Predictions generated for 100 samples
Average generation time: 2.424s
Validating predictions using model-based validator...


Validating: 100%|██████████| 100/100 [00:24<00:00,  4.07it/s]


OVERALL GOLD-CONTEXT EVALUATION RESULTS
Total samples evaluated: 100
Accuracy (Model-Validated): 0.5300 (53/100)
Avg Generation Time: 2.424s
Avg Context Length (chars): 25172.3


In [21]:
import json
from google.colab import files

OUTPUT_JSONL = "simpleqa_gold_context_results.jsonl"

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for row in df_results.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved: {OUTPUT_JSONL}")

files.download(OUTPUT_JSONL)

Saved: simpleqa_gold_context_results.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>